# Bin-Matching Filter

Salvages existing datasets with length confounds by matching length distributions across bins.

**Goal:** Get p > 0.05 (no significant length difference)

## Step 1: Imports

In [ ]:
import json
import numpy as np
from scipy import stats
from collections import Counter

## Step 2: Load Data

**CHANGE `input_file` to your actual JSON file path**

In [ ]:
# CHANGE THIS to your actual file path
input_file = "hotpot_qwen25_rag_length_controlled_1000.json"  # <-- EDIT THIS

with open(input_file, 'r') as f:
    data = json.load(f)

# Split by hallucination status
non_hallucinated = [s for s in data if s['hallucinated'] == 0]
hallucinated = [s for s in data if s['hallucinated'] == 1]

print(f"Loaded {len(non_hallucinated)} non-hallucinated samples")
print(f"Loaded {len(hallucinated)} hallucinated samples")
print(f"Total: {len(data)} samples")

## Step 3: Check BEFORE Bin-Matching

In [ ]:
# Calculate word counts from answer text (if not already present)
if 'word_count' not in non_hallucinated[0]:
    print("word_count field not found, calculating from answer text...")
    for s in non_hallucinated:
        s['word_count'] = len(s['answer'].split())
    for s in hallucinated:
        s['word_count'] = len(s['answer'].split())
    print("✓ Added word_count field\n")

non_hall_lengths = [s['word_count'] for s in non_hallucinated]
hall_lengths = [s['word_count'] for s in hallucinated]

print("=== BEFORE BIN-MATCHING ===")
print(f"Non-hallucinated: {np.mean(non_hall_lengths):.2f} ± {np.std(non_hall_lengths):.2f} words (n={len(non_hallucinated)})")
print(f"Hallucinated:     {np.mean(hall_lengths):.2f} ± {np.std(hall_lengths):.2f} words (n={len(hallucinated)})")
print(f"Difference: {abs(np.mean(non_hall_lengths) - np.mean(hall_lengths)):.2f} words")

t_stat, p_value = stats.ttest_ind(non_hall_lengths, hall_lengths)
print(f"\nT-test: t={t_stat:.3f}, p={p_value:.6f}")

if p_value > 0.05:
    print("✅ Already clean - no filtering needed!")
else:
    print(f"⚠️ Significant difference (p={p_value:.6f}) - needs bin-matching")

## Step 4: Apply Bin-Matching

This creates word-count bins (e.g., 20-22, 22-24, ...) and matches the number of samples in each bin.

In [ ]:
print("\n=== APPLYING BIN-MATCHING ===\n")

# Create bins (2-word intervals)
bins = np.arange(20, 45, 2)  # [20-22, 22-24, 24-26, ..., 42-44]
print(f"Using {len(bins)-1} bins from {bins[0]} to {bins[-1]} words\n")

# Assign each sample to a bin
non_hall_bins = [np.digitize(s['word_count'], bins) for s in non_hallucinated]
hall_bins = [np.digitize(s['word_count'], bins) for s in hallucinated]

# Add bin info temporarily
for i, s in enumerate(non_hallucinated):
    s['bin'] = non_hall_bins[i]
for i, s in enumerate(hallucinated):
    s['bin'] = hall_bins[i]

# Show bin distribution
non_hall_bin_counts = Counter(non_hall_bins)
hall_bin_counts = Counter(hall_bins)

print("Bin distribution BEFORE matching:")
print(f"Non-hallucinated: {dict(sorted(non_hall_bin_counts.items()))}")
print(f"Hallucinated:     {dict(sorted(hall_bin_counts.items()))}\n")

# Match: For each bin, take min(non_hall_count, hall_count)
matched_non_hall = []
matched_hall = []

for bin_id in sorted(set(non_hall_bins + hall_bins)):
    # Get samples in this bin
    non_hall_in_bin = [s for s in non_hallucinated if s['bin'] == bin_id]
    hall_in_bin = [s for s in hallucinated if s['bin'] == bin_id]
    
    # Take minimum count
    min_count = min(len(non_hall_in_bin), len(hall_in_bin))
    
    if min_count > 0:
        # Randomly sample min_count from each
        matched_non_hall.extend(np.random.choice(non_hall_in_bin, min_count, replace=False).tolist())
        matched_hall.extend(np.random.choice(hall_in_bin, min_count, replace=False).tolist())
        print(f"Bin {bin_id}: Kept {min_count} samples from each class")

# Remove bin field
for s in matched_non_hall:
    del s['bin']
for s in matched_hall:
    del s['bin']

# Update lists
non_hallucinated = matched_non_hall
hallucinated = matched_hall

print(f"\nRetained {len(non_hallucinated)} + {len(hallucinated)} = {len(non_hallucinated) + len(hallucinated)} total samples")

## Step 5: Verify Results

In [ ]:
non_hall_lengths = [s['word_count'] for s in non_hallucinated]
hall_lengths = [s['word_count'] for s in hallucinated]

print("\n=== AFTER BIN-MATCHING ===")
print(f"Non-hallucinated: {np.mean(non_hall_lengths):.2f} ± {np.std(non_hall_lengths):.2f} words (n={len(non_hallucinated)})")
print(f"Hallucinated:     {np.mean(hall_lengths):.2f} ± {np.std(hall_lengths):.2f} words (n={len(hallucinated)})")
print(f"Difference: {abs(np.mean(non_hall_lengths) - np.mean(hall_lengths)):.2f} words")

t_stat, p_value = stats.ttest_ind(non_hall_lengths, hall_lengths)
print(f"\nT-test: t={t_stat:.3f}, p={p_value:.6f}")

if p_value > 0.05:
    print("✅ NO significant difference - bin-matching successful!")
else:
    print(f"⚠️ Still significant (p={p_value:.6f}) - may need tighter bins or more data")

# Cohen's d (effect size)
pooled_std = np.sqrt((np.std(non_hall_lengths)**2 + np.std(hall_lengths)**2) / 2)
cohens_d = abs(np.mean(non_hall_lengths) - np.mean(hall_lengths)) / pooled_std
effect_size = 'negligible' if cohens_d < 0.2 else 'small' if cohens_d < 0.5 else 'medium' if cohens_d < 0.8 else 'large'
print(f"\nCohen's d: {cohens_d:.3f} ({effect_size} effect size)")

# VERIFY BIN ALIGNMENT (proof that bins are matched)
print("\n=== BIN ALIGNMENT VERIFICATION ===")
# Recalculate bins for verification
non_hall_bin_verify = [np.digitize(s['word_count'], bins) for s in non_hallucinated]
hall_bin_verify = [np.digitize(s['word_count'], bins) for s in hallucinated]

non_hall_bin_counts_after = Counter(non_hall_bin_verify)
hall_bin_counts_after = Counter(hall_bin_verify)

print("\nPer-bin counts AFTER matching:")
print(f"{'Bin':<6} {'Non-Hall':<10} {'Hall':<10} {'Match?':<8}")
print("-" * 40)
all_bins = sorted(set(non_hall_bin_verify + hall_bin_verify))
perfect_match = True
for b in all_bins:
    n_count = non_hall_bin_counts_after[b]
    h_count = hall_bin_counts_after[b]
    match = "✓" if n_count == h_count else "✗"
    if n_count != h_count:
        perfect_match = False
    print(f"{b:<6} {n_count:<10} {h_count:<10} {match:<8}")

if perfect_match:
    print("\n✅ Perfect bin alignment - all bins have equal counts!")
else:
    print("\n⚠️ Some bins don't match - this shouldn't happen!")

print(f"\n📊 Summary:")
print(f"   - Samples retained: {len(non_hallucinated) + len(hallucinated)} (from {len(data)})")
print(f"   - Retention rate: {100 * (len(non_hallucinated) + len(hallucinated)) / len(data):.1f}%")
print(f"   - p-value: {p_value:.6f} {'✅' if p_value > 0.05 else '❌'}")
print(f"   - Cohen's d: {cohens_d:.3f} ({effect_size})")
print(f"   - Bin alignment: {'Perfect ✅' if perfect_match else 'Imperfect ⚠️'}")

## Step 6: Save Filtered Dataset

In [ ]:
# Combine and shuffle
all_samples = non_hallucinated + hallucinated
np.random.seed(42)
np.random.shuffle(all_samples)

# Remove word_count field (optional - keep if you need it later)
final_samples = [{k: v for k, v in s.items() if k != 'word_count'} for s in all_samples]

# Generate output filename
output_file = input_file.replace('.json', '_binmatched.json')

# Save
with open(output_file, 'w') as f:
    json.dump(final_samples, f, indent=2)

print(f"\n✅ Saved filtered dataset to: {output_file}")
print(f"   Total samples: {len(final_samples)}")
print(f"   Non-hallucinated: {sum(1 for s in final_samples if s['hallucinated'] == 0)}")
print(f"   Hallucinated: {sum(1 for s in final_samples if s['hallucinated'] == 1)}")

## Summary

This notebook applies **bin-matching** to remove length confounds:

1. Creates word-count bins (20-22, 22-24, ...)
2. For each bin, keeps equal number of hallucinated and non-hallucinated samples
3. Results in identical length distributions
4. Achieves p > 0.05 (no significant difference)

**Next steps:**
- Use the `_binmatched.json` file for uncertainty extraction
- Report in paper: "We applied bin-matching to ensure no length confound (p > 0.05)"